# 01 – Data Collection & Cleaning (Rivindu)

This notebook builds the real dataset for the AI Job & Skill Matcher:

1. Download Kaggle **LinkedIn Job Postings** and keep entry-level / internship jobs
2. Load the **Sri Lankan job ads** the team collected in Google Sheets
3. Merge, clean and de-duplicate → `jobs_clean.csv`
4. Build the skill list from **ESCO** → `skills.csv`
5. Extract skills for every job → `job_skills.csv`
6. Quality checks

Run the cells from top to bottom. Outputs go to `data/processed/`.

## 0. Setup

In [ ]:
# Clone the repo (first time only) and move to your branch
!git clone https://github.com/<leader-username>/ai-job-skill-matcher.git
%cd ai-job-skill-matcher
!git checkout feature/data-cleaning
!pip install -q -r requirements.txt kagglehub

In [ ]:
import re, os, glob
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)
OUT = Path("data/processed"); OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)

## 1. Kaggle – LinkedIn Job Postings

Dataset: https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

`kagglehub` downloads it straight into Colab. If it asks for login, go to Kaggle → Settings → **Create New Token**, and upload `kaggle.json`; or download the zip manually and upload `postings.csv` to `data/raw/`.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("arshkon/linkedin-job-postings")
postings_file = glob.glob(f"{path}/**/postings.csv", recursive=True)[0]
li = pd.read_csv(postings_file)
print(li.shape)
li.columns.tolist()

In [ ]:
# Keep only jobs a university student can apply for
LEVELS = ["Entry level", "Internship"]
TITLE_HINTS = r"\b(?:intern|internship|trainee|graduate|junior|entry[- ]level|associate)\b"

lvl_col = "formatted_experience_level" if "formatted_experience_level" in li.columns else None
mask = li["title"].str.contains(TITLE_HINTS, case=False, na=False)
if lvl_col:
    mask |= li[lvl_col].isin(LEVELS)
li_entry = li[mask].copy()
print("entry-level jobs:", len(li_entry))

# Optional: keep the dataset small enough for free hosting
li_entry = li_entry.sample(min(len(li_entry), 2500), random_state=42)

In [ ]:
def to_job_type(row):
    wt = str(row.get("formatted_work_type", "")).lower()
    lvl = str(row.get(lvl_col, "")).lower() if lvl_col else ""
    title = str(row.get("title", "")).lower()
    if "intern" in wt or "intern" in lvl or "intern" in title:
        return "Internship"
    if "part" in wt:
        return "Part-time"
    return "Full-time"

li_jobs = pd.DataFrame({
    "title": li_entry["title"],
    "description": li_entry["description"].fillna("") + " " + li_entry.get("skills_desc", pd.Series("", index=li_entry.index)).fillna(""),
    "job_type": li_entry.apply(to_job_type, axis=1),
    "location": li_entry.get("location", "Not stated"),
    "source": "Kaggle LinkedIn",
    "collected_date": "2023-2024",
})
li_jobs.head()

## 2. Sri Lankan job ads (team Google Sheet)

In the Google Sheet: **File → Share → Publish to web → CSV**, copy the link and paste it below.
(Or **File → Download → CSV** and upload it to `data/raw/sl_jobs.csv`.)

Columns must follow `sl_jobs_template.csv`.

In [ ]:
SHEET_CSV_URL = ""   # paste the published CSV link here

if SHEET_CSV_URL:
    sl = pd.read_csv(SHEET_CSV_URL)
else:
    sl = pd.read_csv(RAW / "sl_jobs.csv")
sl = sl.dropna(how="all")
print("Sri Lankan ads:", len(sl))
print(sl["collected_by"].value_counts() if "collected_by" in sl else "")

In [ ]:
# Put the required skills column into the description so skill extraction sees it
sl_jobs = pd.DataFrame({
    "title": sl["title"],
    "description": sl["description"].fillna("") + " Required skills: " + sl["required_skills"].fillna(""),
    "job_type": sl["job_type"],
    "field": sl.get("field"),
    "location": sl["location"].fillna("Not stated"),
    "source": sl["source"],
    "collected_date": sl["collected_date"],
})

### 2b. Translate Sinhala ads (uses Hijaz's module)
Only rows with Sinhala text are sent to Gemini, in small batches to stay inside the free-tier limits.

In [ ]:
import time
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
from core.language import detect_language, translate_si_en

def translate_if_sinhala(text):
    if detect_language(str(text)) == "si":
        time.sleep(4)  # free tier rate limit
        return translate_si_en(str(text))[0]
    return text

for col in ["title", "description"]:
    sl_jobs[col] = sl_jobs[col].apply(translate_if_sinhala)

## 3. Merge and clean

In [ ]:
FIELD_RULES = {
    "IT": r"software|developer|engineer.*(software|data|ml|qa)|data|it |network|web|cyber|devops|qa|ui|ux|programmer|analyst.*(data|system)",
    "Business": r"account|finance|business|marketing|sales|hr|human resource|bank|audit|admin|management|procurement|logistics",
    "Engineering": r"civil|mechanical|electrical|electronic|chemical|manufactur|quantity survey|site engineer",
    "Science": r"lab|research|chemist|biolog|scien|medical|pharma|environment",
    "Arts": r"design|graphic|content|writer|media|teacher|tutor|translator|journalis|creative",
}
def guess_field(title):
    t = f" {str(title).lower()} "
    for field, pattern in FIELD_RULES.items():
        if re.search(pattern, t):
            return field
    return "Other"

jobs = pd.concat([li_jobs, sl_jobs], ignore_index=True)
jobs["field"] = jobs["field"].fillna(jobs["title"].apply(guess_field)) if "field" in jobs else jobs["title"].apply(guess_field)

# normalise enums
def norm_type(t):
    t = str(t).lower()
    if "intern" in t: return "Internship"
    if "part" in t: return "Part-time"
    return "Full-time"
jobs["job_type"] = jobs["job_type"].apply(norm_type)
jobs.loc[~jobs["job_type"].isin(["Internship", "Full-time", "Part-time"]), "job_type"] = "Full-time"
jobs.loc[~jobs["field"].isin(["IT", "Business", "Engineering", "Science", "Arts", "Other"]), "field"] = "Other"

# drop empty and duplicate rows
before = len(jobs)
jobs = jobs.dropna(subset=["title", "description"])
jobs = jobs[jobs["description"].str.len() > 50]
jobs["_key"] = jobs["title"].str.lower().str.strip() + "|" + jobs["description"].str.lower().str[:300]
jobs = jobs.drop_duplicates("_key").drop(columns="_key")
print(f"{before} -> {len(jobs)} rows after removing empty / duplicate jobs")

In [ ]:
from core.preprocess import clean_text

jobs["title"] = jobs["title"].str.strip()
jobs["description"] = jobs["description"].apply(clean_text)
jobs = jobs.reset_index(drop=True)
jobs.insert(0, "job_id", [f"J{i:04d}" for i in range(1, len(jobs) + 1)])
jobs["location"] = jobs["location"].fillna("Not stated")

jobs.to_csv(OUT / "jobs_clean.csv", index=False)
jobs["field"].value_counts()

## 4. Skill list from ESCO

Download from https://esco.ec.europa.eu/en/use-esco/download → **CSV, English** → unzip and upload `skills_en.csv` to `data/raw/`.

Keeping all ~13,000 ESCO skills makes matching slow and noisy, so we keep the ~400 that actually appear in our job ads, plus our own skills (soft skills, Sri Lankan terms).

In [ ]:
esco = pd.read_csv(RAW / "skills_en.csv")
esco = esco[["preferredLabel", "altLabels", "skillType"]].dropna(subset=["preferredLabel"])
print(len(esco), "ESCO skills")

def labels(row):
    out = [row.preferredLabel]
    if isinstance(row.altLabels, str):
        out += [a for a in row.altLabels.split("\n") if a.strip()]
    return [l.strip().lower() for l in out if 2 <= len(l.strip()) <= 40]

esco["labels"] = esco.apply(labels, axis=1)

In [ ]:
# Fast counting: build the set of 1-3 word phrases in every job, then look labels up
def ngrams(text, n_max=3):
    words = re.findall(r"[a-z0-9+#.]+", text.lower())
    grams = set()
    for n in range(1, n_max + 1):
        grams.update(" ".join(words[i:i+n]) for i in range(len(words) - n + 1))
    return grams

job_grams = [ngrams(t) for t in jobs["description"]]
from collections import Counter
doc_freq = Counter(g for grams in job_grams for g in grams)

esco["jobs_with_skill"] = esco["labels"].apply(lambda ls: max((doc_freq.get(l, 0) for l in ls), default=0))
top = esco[esco["jobs_with_skill"] >= 5].sort_values("jobs_with_skill", ascending=False).head(400)
top[["preferredLabel", "jobs_with_skill"]].head(30)

In [ ]:
# Look through the list and remove useless very common words here
DROP = {"english", "communication", "management", "planning", "research"}  # kept from our own list instead
top = top[~top["preferredLabel"].str.lower().isin(DROP)]

existing = pd.read_csv(OUT / "skills.csv")          # our hand-made skills (soft skills etc.)
# every name + alias we already have, so ESCO does not add duplicates (e.g. "excel")
existing_terms = set(existing.skill_name.str.lower())
for a in existing.aliases.dropna():
    existing_terms.update(x.strip().lower() for x in a.split(";") if x.strip())

new_rows = []
next_id = len(existing) + 1
for r in top.itertuples():
    name = re.sub(r"\s*\(.*?\)", "", r.preferredLabel).strip()   # "Python (computer programming)" -> "Python"
    if name.lower() in existing_terms or existing_terms & set(r.labels):   # already covered
        continue
    aliases = ";".join(sorted(set(r.labels) - {name.lower()})[:8])
    existing_terms.update(r.labels + [name.lower()])
    category = "Technical"
    new_rows.append({"skill_id": f"S{next_id:03d}", "skill_name": name[:1].upper() + name[1:],
                     "category": category, "aliases": aliases})
    next_id += 1

skills = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
skills.to_csv(OUT / "skills.csv", index=False)
print(len(skills), "skills saved")

## 5. Skills for every job → `job_skills.csv`

Uses Mathu's keyword extractor (no Gemini, so it is fast and free).
A skill is **Required** if it appears in a sentence with words like *must, required, essential*; otherwise **Preferred**.

In [ ]:
from core import preprocess, skill_extractor
from utils import data_loader
for f in [preprocess.alias_map, data_loader.load_skills]:
    f.cache_clear()      # reload the new skills.csv

name_to_id = dict(zip(skills.skill_name, skills.skill_id))
REQ = re.compile(r"must|required|requirement|essential|mandatory", re.I)

rows = []
for job in jobs.itertuples():
    found = skill_extractor.extract_keywords(job.description)
    req_text = " ".join(s for s in re.split(r"[.!?\n]", job.description) if REQ.search(s))
    required = skill_extractor.extract_keywords(req_text) if req_text else set()
    for s in found:
        rows.append({"job_id": job.job_id, "skill_id": name_to_id[s],
                     "importance": "Required" if s in required else "Preferred"})

job_skills = pd.DataFrame(rows)
job_skills.to_csv(OUT / "job_skills.csv", index=False)
print(len(job_skills), "job-skill pairs")

## 6. Quality checks (targets from the Data Acquisition plan)

In [ ]:
per_job = job_skills.groupby("job_id").size().reindex(jobs.job_id, fill_value=0)
report = {
    "total jobs": len(jobs),
    "Sri Lankan jobs": int((jobs.source != "Kaggle LinkedIn").sum()),
    "duplicate rows": int(jobs.duplicated(["title", "description"]).sum()),
    "empty descriptions": int((jobs.description.str.len() == 0).sum()),
    "% jobs with >= 3 skills": round((per_job >= 3).mean() * 100, 1),
    "skills in list": len(skills),
}
report

In [ ]:
# Manual check: look at 5% of jobs and their skills. Error rate should be <= 10%.
sample = jobs.sample(max(1, len(jobs) // 20), random_state=1)
for j in sample.head(10).itertuples():
    s = job_skills[job_skills.job_id == j.job_id].merge(skills, on="skill_id").skill_name.tolist()
    print(f"{j.job_id} | {j.title}\n   -> {s}\n")

## 7. Save to GitHub

Large files (over 50 MB) must go to Google Drive instead.
After pushing, tell **Yoosuf** to rebuild the embeddings (`python -m scripts.build_embeddings`).

In [ ]:
!du -h data/processed/*.csv
!git add data/processed/jobs_clean.csv data/processed/skills.csv data/processed/job_skills.csv notebooks/01_data_cleaning.ipynb
!git -c user.name="Rivindu" -c user.email="you@example.com" commit -m "Add cleaned job dataset, ESCO skills and job skills"
# In Colab, push with a GitHub token:  !git push https://<TOKEN>@github.com/<leader-username>/ai-job-skill-matcher.git feature/data-cleaning